Sprint 1 (Importação dos dados)

In [8]:
import numpy as np
import pandas as pd
import re
import matplotlib as plt
from IPython.display import display
from datetime import datetime


#Importando o arquivo BaseVarejo.csv
df_vendas = pd.read_csv("BaseVarejo/BaseVarejo.csv", encoding='utf-8', sep=';')


In [ ]:
# Exibição dos dados básicos (nº de registro, linhas, colunas e tipos de dados) 
print(f"Número de registros: {len(df_vendas)}")
print(f"Número de linhas e colunas: {df_vendas.shape}")

print("\nPrimeiras 5 linhas:")
display(df_vendas.head())
display(df_vendas.tail())
df_vendas.info()


### SPRINT 2: DIAGNÓSTICO E FUNÇÕES DE LIMPEZA

In [54]:
#1.Limpeza das colunas vazias (encontradas 4 colunas com o nome Unnamed)
df_limpo = df_vendas.drop(columns=[col for col in df_vendas.columns if 'Unnamed' in col])
print(f"Colunas vazias ('Unnamed') eliminadas: {len(df_limpo.columns) - len(df_vendas.columns)}")

display(df_limpo.tail())

Colunas vazias ('Unnamed') eliminadas: -4


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
829995,19/08/2022,919822,155,F,2,0,B,183,ALIMENTOS,KETCHUP
829996,19/08/2022,919822,155,F,2,0,B,56,ALIMENTOS,QUEIJO MUSSARELA
829997,19/08/2022,919822,155,F,2,0,B,227,ALIMENTOS,ARROZ
829998,19/08/2022,919822,155,F,2,0,B,214,ALIMENTOS,CEBOLA
829999,19/08/2022,919822,155,F,2,0,B,59,ALIMENTOS,SALGADINHO


In [ ]:
### Inconsistências de dados

#2. Procurar valores ausentes 
ausentes = df_limpo.isna().sum()
print(f"Número de valores ausentes por coluna:\n{ausentes}")


### SPRINT 3: LIMPEZA E TRATAMENTO DOS DADOS

In [56]:
#1. Identificando duplicidades
print(f"Número de registros duplicados: {df_limpo.duplicated().sum()}")
# Remoção de duplicatas exatas
df_limpo = df_limpo.drop_duplicates().reset_index(drop=True)
display(df_limpo.shape)

Número de registros duplicados: 96553


(733447, 10)

In [ ]:
## Verificando valores ausentes na coluna CL_FHL (nº de filhos)
display(df_limpo['CL_FHL'].describe(), df_limpo['CL_FHL'].value_counts())
ausentes_filhos =df_limpo['CL_FHL'].isna().sum()
print(f"Número de valores ausentes na coluna CL_FHL: {ausentes_filhos}")

count    733447.000000
mean          1.146049
std           1.416917
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max           4.000000
Name: CL_FHL, dtype: float64

CL_FHL
0    384986
2     94168
3     92407
1     90845
4     71041
Name: count, dtype: int64

Número de valores ausentes na coluna CL_FHL: 0


In [76]:
#2. Detectando Outliers na coluna CL_FHL (nº de filhos) usando o método IQR
Q1 = df_limpo['CL_FHL'].quantile(0.25)
Q3 = df_limpo['CL_FHL'].quantile(0.75)
IQR = Q3 - Q1

limite_superior = Q3 + 1.5 * IQR
print(f"Limite superior: {limite_superior}")

#contar registro acima do limite superior
limite_superior_contagem = df_limpo[df_limpo['CL_FHL']> limite_superior]
print(f"Total de registro acima do limite: {len(limite_superior_contagem)}")


Limite superior: 5.0
Total de registro acima do limite: 0


In [83]:
#CL_FHL (coluna nº de filhos) - utilizarei a mediana para não sofrer com outliers
status_filhos = df_limpo['CL_FHL']

status_filhos ={
  "Contagem de filhos": filhos.value_counts(),
  "Média": filhos.mean(),   
  "Mediana": filhos.median(),
  "Desvio Padrão": filhos.std(),    
  "Mínimo": filhos.min(),
  "Máximo": filhos.max(),
  "1º Quartil": filhos.quantile(0.25),
  "3º Quartil": filhos.quantile(0.75),
}

df_stats = pd.DataFrame(
    list(status_filhos.items()), columns=['Parâmetro Estatístico', 'Valor']
)
print(df_stats.to_string(index=False))

Parâmetro Estatístico                                                                                        Valor
   Contagem de filhos CL_FHL
0    384986
2     94168
3     92407
1     90845
4     71041
Name: count, dtype: int64
                Média                                                                                     1.146049
              Mediana                                                                                          0.0
        Desvio Padrão                                                                                     1.416917
               Mínimo                                                                                            0
               Máximo                                                                                            4
           1º Quartil                                                                                          0.0
           3º Quartil                                                           

In [84]:
# Agrupamento 1: Total de Compras e Média de Filhos por Gênero (Groupby)
agrup_genero = (
    df_limpo.groupby("CL_GENERO")
    .agg(
        Total_Transacoes=("CO_ID", "count"),
        Media_Filhos=("CL_FHL", "mean"),
    )
    .reset_index()
)

print("\n--- Agrupamento 1: Padrão de Compras por Gênero (groupby) ---")
print(agrup_genero.to_string(index=False))


--- Agrupamento 1: Padrão de Compras por Gênero (groupby) ---
CL_GENERO  Total_Transacoes  Media_Filhos
        F            382427      1.089562
        M            351020      1.207589
